# ORI 2026 - Valute: Vizualizacija rezultata

Ovaj notebook prikazuje grafikone za:
1. Cene valutnih parova tokom vremena
2. Tehnički indikatori za jedan par
3. Rezultati NN eksperimenata (R², Accuracy, MAE, F1)


In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings("ignore")

# Stil
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#f8f8f8",
    "axes.grid":        True,
    "grid.alpha":       0.4,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "font.size":        11,"text.color":       "black",
    "axes.labelcolor":  "black",
    "xtick.color":      "black",
    "ytick.color":      "black",
})

PAIRS   = ["EURUSD", "USDCAD", "USDCHF", "USDGBP", "USDJPY"]
COLORS  = ["#2196F3", "#E91E63", "#4CAF50", "#FF9800", "#9C27B0"]
PAIR_COLOR = dict(zip(PAIRS, COLORS))

print("Paketi učitani.")


Paketi učitani.


## Učitavanje podataka

In [14]:
currency_all  = pd.read_excel("Currency Data ALL.xlsx",      sheet_name="Currency_Data_ALL")
features      = pd.read_excel("Currency Data Features.xlsx")
results       = pd.read_excel("NN_Experiment_Results.xlsx",  sheet_name="All_Results")
missing_summ  = pd.read_excel("Currency Data ALL.xlsx",      sheet_name="Missing_Summary")

currency_all["Date"] = pd.to_datetime(currency_all["Date"])
features["Date"]     = pd.to_datetime(features["Date"])

reg = results[results["Problem"] == "regression"].copy()
cls = results[results["Problem"] == "classification"].copy()

print(f"currency_all : {currency_all.shape}")
print(f"features     : {features.shape}")
print(f"results      : {results.shape}  ({len(reg)} regresija, {len(cls)} klasifikacija)")


currency_all : (13050, 3)
features     : (13050, 39)
results      : (85, 16)  (65 regresija, 20 klasifikacija)


## Grafikon 1 — Kretanje cena svih valutnih parova (2016–2026)

Svaki valutni par normalizujemo na 100 na prvom danu da bi se mogli
porediti na istoj osi (jer su npr. USDJPY ~100+ a EURUSD ~1.0).


In [15]:
fig, axes = plt.subplots(len(PAIRS), 1, figsize=(14, 12), sharex=True)

for ax, pair, color in zip(axes, PAIRS, COLORS):
    sub = currency_all[currency_all["Pair"] == pair].dropna(subset=["Price"])
    ax.plot(sub["Date"], sub["Price"], color=color, linewidth=1.2, label=pair)
    ax.set_ylabel(pair, fontsize=10, rotation=0, labelpad=55)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.4f"))
    ax.legend(loc="upper left", fontsize=9)

axes[-1].set_xlabel("Datum")
fig.suptitle("Kretanje cena valutnih parova (2016–2026)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("plot1_cene.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sačuvano: plot1_cene.png")


Sačuvano: plot1_cene.png


## Grafikon 2 — Normalizovane cene (baza = 100 na prvom danu)

Svi parovi na istoj osi — vidimo ko je "porastao" ili "pao" relativno
u posmatranom periodu.


In [16]:
fig, ax = plt.subplots(figsize=(14, 5))

for pair, color in PAIR_COLOR.items():
    sub = currency_all[currency_all["Pair"] == pair].dropna(subset=["Price"]).copy()
    sub = sub.sort_values("Date")
    normalized = sub["Price"] / sub["Price"].iloc[0] * 100
    ax.plot(sub["Date"], normalized, color=color, linewidth=1.4, label=pair)

ax.axhline(100, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
ax.set_title("Normalizovane cene valutnih parova (100 = početna vrednost)", fontsize=13, fontweight="bold")
ax.set_ylabel("Indeks (početak = 100)")
ax.set_xlabel("Datum")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("plot2_normirane_cene.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sačuvano: plot2_normirane_cene.png")


Sačuvano: plot2_normirane_cene.png


## Grafikon 3 — Tehnički indikatori za EURUSD (poslednja 2 godine)

Prikazujemo: Cena + SMA/EMA, RSI, MACD i STD (volatilnost).
Ovo je klasičan prikaz kakav se koristi u tehničkoj analizi.


In [17]:
pair = "EURUSD"
sub = features[features["Pair"] == pair].copy().sort_values("Date")
sub = sub[sub["Date"] >= "2024-01-01"]   # poslednje 2 godine

fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

# Panel 1: Cena + SMA + EMA
axes[0].plot(sub["Date"], sub["Price"],   color="#2196F3", linewidth=1.2, label="Price")
axes[0].plot(sub["Date"], sub["SMA"],     color="#FF9800", linewidth=1.2, label="SMA(14)", linestyle="--")
axes[0].plot(sub["Date"], sub["EMA"],     color="#E91E63", linewidth=1.2, label="EMA(14)", linestyle=":")
axes[0].set_title(f"{pair} — Cena sa SMA i EMA", fontweight="bold")
axes[0].set_ylabel("Cena (USD)")
axes[0].legend(fontsize=9)

# Panel 2: RSI
axes[1].plot(sub["Date"], sub["RSI"], color="#9C27B0", linewidth=1.2)
axes[1].axhline(70, color="red",   linestyle="--", alpha=0.7, linewidth=0.9, label="Precenjenost (70)")
axes[1].axhline(30, color="green", linestyle="--", alpha=0.7, linewidth=0.9, label="Potcenjenost (30)")
axes[1].fill_between(sub["Date"], sub["RSI"], 70, where=(sub["RSI"] >= 70), alpha=0.2, color="red")
axes[1].fill_between(sub["Date"], sub["RSI"], 30, where=(sub["RSI"] <= 30), alpha=0.2, color="green")
axes[1].set_ylim(0, 100)
axes[1].set_title("RSI(14)", fontweight="bold")
axes[1].set_ylabel("RSI")
axes[1].legend(fontsize=9)

# Panel 3: MACD
axes[2].plot(sub["Date"], sub["MACD"],        color="#2196F3", linewidth=1.1, label="MACD")
axes[2].plot(sub["Date"], sub["MACD_signal"], color="#FF9800", linewidth=1.1, label="Signal", linestyle="--")
axes[2].bar( sub["Date"], sub["MACD_hist"],   color=np.where(sub["MACD_hist"] >= 0, "#4CAF50", "#F44336"),
             alpha=0.5, label="Histogram", width=1)
axes[2].axhline(0, color="black", linewidth=0.7)
axes[2].set_title("MACD(12,26,9)", fontweight="bold")
axes[2].set_ylabel("Vrednost")
axes[2].legend(fontsize=9)

# Panel 4: STD (volatilnost)
axes[3].fill_between(sub["Date"], sub["STD"], alpha=0.4, color="#607D8B")
axes[3].plot(sub["Date"], sub["STD"], color="#607D8B", linewidth=1.1)
axes[3].set_title("STD(14) — Pokretna standardna devijacija (volatilnost)", fontweight="bold")
axes[3].set_ylabel("STD")
axes[3].set_xlabel("Datum")

plt.tight_layout()
plt.savefig("plot3_indikatori_eurusd.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sačuvano: plot3_indikatori_eurusd.png")


Sačuvano: plot3_indikatori_eurusd.png


## Grafikon 4 — R² po eksperimentu (regresija: predviđanje CENE)

R² blizu 1.0 = model dobro predvida.
R² blizu 0 ili negativan = model ne predvida bolje od proseka.

Samo eksperimenti grupe 1 (1-1a/b/c) predviđaju apsolutnu cenu.


In [18]:
price_reg = reg[reg["Target"].str.startswith("Target_Price")].copy()

if not price_reg.empty:
    pivot = price_reg.groupby(["Experiment", "Pair"])["R2"].mean().unstack(fill_value=np.nan)

    fig, ax = plt.subplots(figsize=(11, 5))
    x = np.arange(len(pivot.index))
    width = 0.15
    for i, (pair, color) in enumerate(PAIR_COLOR.items()):
        if pair in pivot.columns:
            ax.bar(x + i * width, pivot[pair], width, label=pair, color=color, alpha=0.85)

    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(pivot.index, fontsize=10)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.axhline(1, color="green", linewidth=0.8, linestyle=":", alpha=0.5)
    ax.set_title("R² po eksperimentu i valutnom paru (predviđanje cene)", fontsize=13, fontweight="bold")
    ax.set_ylabel("R² (prosek po horizontima)")
    ax.set_xlabel("Eksperiment")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("plot4_r2_cena.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Sačuvano: plot4_r2_cena.png")
else:
    print("Nema podataka za predviđanje cene.")


Sačuvano: plot4_r2_cena.png


## Grafikon 5 — R² po horizontu predviđanja (regresija: prinos)

Što je horizont duži (t+10 vs t+1), to je predviđanje teže.
Eksperimenti grupe 2 predviđaju **prinos (ROC)** umesto apsolutne cene.


In [19]:
return_reg = reg[reg["Target"].str.startswith("Target_Return")].copy()

if not return_reg.empty:
    pivot_h = return_reg.groupby(["Experiment", "Horizon"])["R2"].mean().unstack(fill_value=np.nan)

    fig, ax = plt.subplots(figsize=(11, 5))
    x = np.arange(len(pivot_h.index))
    horizon_colors = ["#1E88E5","#43A047","#FB8C00","#E53935"]
    horizons = sorted(return_reg["Horizon"].unique())
    width = 0.18

    for i, (h, color) in enumerate(zip(horizons, horizon_colors)):
        if h in pivot_h.columns:
            ax.bar(x + i * width, pivot_h[h], width, label=f"t+{h}", color=color, alpha=0.85)

    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(pivot_h.index, fontsize=10)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("R² po eksperimentu i horizontu (predviđanje prinosa)", fontsize=13, fontweight="bold")
    ax.set_ylabel("R² (prosek po parovima)")
    ax.set_xlabel("Eksperiment")
    ax.legend(title="Horizont", fontsize=9)
    plt.tight_layout()
    plt.savefig("plot5_r2_prinos_horizont.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Sačuvano: plot5_r2_prinos_horizont.png")
else:
    print("Nema podataka za predviđanje prinosa.")


Sačuvano: plot5_r2_prinos_horizont.png


## Grafikon 6 — Accuracy po valutnom paru (klasifikacija: smer prinosa)

Predviđamo da li će cena porasti ili pasti (1 ili 0).
**50% = random** — koliko bolji od bacanja novčića smo?


In [20]:
if not cls.empty:
    pivot_cls = cls.groupby(["Pair", "Horizon"])["Accuracy"].mean().unstack(fill_value=np.nan)

    fig, ax = plt.subplots(figsize=(11, 5))
    x = np.arange(len(pivot_cls.index))
    horizon_colors = ["#1E88E5","#43A047","#FB8C00","#E53935"]
    horizons = sorted(cls["Horizon"].unique())
    width = 0.18

    for i, (h, color) in enumerate(zip(horizons, horizon_colors)):
        if h in pivot_cls.columns:
            ax.bar(x + i * width, pivot_cls[h], width, label=f"t+{h}", color=color, alpha=0.85)

    ax.axhline(0.5, color="red", linestyle="--", linewidth=1.2, label="Random (50%)")
    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(pivot_cls.index, fontsize=10)
    ax.set_ylim(0.35, 0.75)
    ax.set_title("Accuracy klasifikacije po valutnom paru i horizontu", fontsize=13, fontweight="bold")
    ax.set_ylabel("Accuracy")
    ax.set_xlabel("Valutni par")
    ax.legend(title="Horizont", fontsize=9)
    plt.tight_layout()
    plt.savefig("plot6_accuracy.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Sačuvano: plot6_accuracy.png")
else:
    print("Nema klasifikacionih rezultata.")


Sačuvano: plot6_accuracy.png


## Grafikon 7 — F1 Score po eksperimentu (klasifikacija)

F1 kombinuje Precision i Recall. Bolji od Accuracy kada su klase
nebalansirane (npr. više dana rasta nego pada ili obrnuto).


In [21]:
if not cls.empty:
    f1_pivot = cls.groupby(["Experiment", "Pair"])["F1"].mean().unstack(fill_value=np.nan)

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(f1_pivot.index))
    width = 0.15

    for i, (pair, color) in enumerate(PAIR_COLOR.items()):
        if pair in f1_pivot.columns:
            ax.bar(x + i * width, f1_pivot[pair], width, label=pair, color=color, alpha=0.85)

    ax.axhline(0.5, color="red", linestyle="--", linewidth=1.0, label="Baseline (0.5)")
    ax.set_xticks(x + width * 2)
    ax.set_xticklabels(f1_pivot.index, fontsize=10)
    ax.set_title("F1 Score po eksperimentu i valutnom paru (klasifikacija)", fontsize=13, fontweight="bold")
    ax.set_ylabel("F1 Score")
    ax.set_xlabel("Eksperiment")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("plot7_f1.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Sačuvano: plot7_f1.png")


Sačuvano: plot7_f1.png


## Grafikon 8 — MAE (Mean Absolute Error) po valutnom paru

MAE = prosečna apsolutna greška u predviđanju.
Za EURUSD cenu od ~1.08, MAE=0.005 znači greška od 0.5%.


In [22]:
price_reg2 = reg[reg["Target"].str.startswith("Target_Price")].copy()

if not price_reg2.empty:
    mae_pivot = price_reg2.groupby(["Pair", "Horizon"])["MAE"].mean().unstack(fill_value=np.nan)

    fig, ax = plt.subplots(figsize=(11, 5))
    x = np.arange(len(mae_pivot.index))
    horizon_colors = ["#1E88E5","#43A047","#FB8C00","#E53935"]
    horizons = sorted(price_reg2["Horizon"].unique())
    width = 0.2

    for i, (h, color) in enumerate(zip(horizons, horizon_colors)):
        if h in mae_pivot.columns:
            ax.bar(x + i * width, mae_pivot[h], width, label=f"t+{h}", color=color, alpha=0.85)

    ax.set_xticks(x + width * 1.5)
    ax.set_xticklabels(mae_pivot.index, fontsize=10)
    ax.set_title("MAE po valutnom paru i horizontu (predviđanje cene)", fontsize=13, fontweight="bold")
    ax.set_ylabel("MAE (prosek)")
    ax.set_xlabel("Valutni par")
    ax.legend(title="Horizont", fontsize=9)
    plt.tight_layout()
    plt.savefig("plot8_mae.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Sačuvano: plot8_mae.png")


Sačuvano: plot8_mae.png


## Grafikon 9 — Heatmapa R² (predviđanje cene)

Tamnija boja = bolji R². Vizualno odmah vidiš koji par/horizont
model najlakše/najteže predvida.


In [23]:
price_reg3 = reg[reg["Target"].str.startswith("Target_Price")].copy()

if not price_reg3.empty:
    heat_data = price_reg3.groupby(["Pair", "Horizon"])["R2"].mean().unstack()

    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(heat_data.values, cmap="RdYlGn", aspect="auto", vmin=-0.5, vmax=1.0)
    plt.colorbar(im, ax=ax, label="R²")

    ax.set_xticks(range(len(heat_data.columns)))
    ax.set_xticklabels([f"t+{h}" for h in heat_data.columns], fontsize=11)
    ax.set_yticks(range(len(heat_data.index)))
    ax.set_yticklabels(heat_data.index, fontsize=11)

    for i in range(len(heat_data.index)):
        for j in range(len(heat_data.columns)):
            val = heat_data.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                        fontsize=11, color="black", fontweight="bold")

    ax.set_title("Heatmapa R² (predviđanje cene, prosek po eksperimentima)", fontsize=13, fontweight="bold")
    ax.set_xlabel("Horizont predviđanja")
    ax.set_ylabel("Valutni par")
    plt.tight_layout()
    plt.savefig("plot9_heatmapa_r2.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Sačuvano: plot9_heatmapa_r2.png")


Sačuvano: plot9_heatmapa_r2.png


## Grafikon 10 — Distribucija dnevnih prinosa po valutnom paru

Histogrami pokazuju koliko su prinosi "razbacani" (volatilnost).
Uski i visoki histogram = stabilan par; širok = volatilan.


In [24]:
fig, axes = plt.subplots(1, len(PAIRS), figsize=(16, 4), sharey=False)

for ax, pair, color in zip(axes, PAIRS, COLORS):
    sub = features[features["Pair"] == pair]["Price_ROC"].dropna()
    ax.hist(sub, bins=60, color=color, alpha=0.75, edgecolor="white", linewidth=0.3)
    ax.axvline(0,         color="black", linewidth=1.0, linestyle="--")
    ax.axvline(sub.mean(), color="red",  linewidth=1.2, linestyle="-",  label=f"mean={sub.mean():.3f}")
    ax.set_title(pair, fontweight="bold")
    ax.set_xlabel("Dnevni prinos (%)")
    ax.legend(fontsize=8)
    if ax == axes[0]:
        ax.set_ylabel("Frekvencija")

fig.suptitle("Distribucija dnevnih prinosa (ROC) po valutnom paru", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("plot10_distribucija_prinosa.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sačuvano: plot10_distribucija_prinosa.png")


Sačuvano: plot10_distribucija_prinosa.png
